# Admin API Testing and UI/UX Evaluation

This notebook documents the testing of admin functionalities for the Shelf Stacker Admin interface. We'll focus on:

1. Authentication setup
2. Voucher management testing
3. Order management testing
4. API response validation
5. UI/UX evaluation

## Environment Setup
First, we'll set up our testing environment and authentication.

In [ ]:
import requests
import json
from datetime import datetime, timedelta

# API Configuration
API_BASE_URL = 'http://localhost:3000/api'  # Adjust as needed
headers = {
    'Content-Type': 'application/json',
    'Authorization': None  # Will be set after authentication
}

def setup_auth_token():
    """Get admin token from environment or generate new one"""
    import os
    token = os.getenv('TEST_ADMIN_TOKEN')
    if not token:
        print("No TEST_ADMIN_TOKEN found. Please follow these steps:")
        print("1. Set PHUONGDUYNGUYEN_SECRET environment variable")
        print("2. Run: node scripts/generate-test-token.js")
        print("3. Set TEST_ADMIN_TOKEN environment variable")
        return False
    
    headers['Authorization'] = f'Bearer {token}'
    return True

# Initialize authentication
auth_success = setup_auth_token()
print(f"Authentication setup: {'Success' if auth_success else 'Failed'}")

# Voucher Management Testing

We'll test the following voucher management endpoints:
- GET /vouchers (list all vouchers)
- POST /vouchers (create new voucher)
- GET /vouchers/admin/:id (get specific voucher)
- PUT /vouchers/admin/:id (update voucher)
- DELETE /vouchers/admin/:id (soft delete voucher)

First, let's create test helper functions for voucher operations:

In [ ]:
def test_voucher_management():
    # Test voucher creation
    test_voucher = {
        "voucher_id": "TEST123",
        "voucher_type": "discount",
        "discount_type": "fixed",
        "discount_value": 50000,
        "min_order_value": 100000,
        "usage_limit": 100,
        "max_per_user": 1,
        "start_date": (datetime.now()).isoformat(),
        "end_date": (datetime.now() + timedelta(days=7)).isoformat(),
        "description": "Test voucher for API validation"
    }
    
    # Create voucher
    create_response = requests.post(
        f"{API_BASE_URL}/vouchers",
        headers=headers,
        json=test_voucher
    )
    print("Create voucher response:", create_response.status_code)
    if create_response.status_code == 201:
        created_voucher = create_response.json()
        voucher_id = created_voucher.get('_id')
        
        # Test get voucher
        get_response = requests.get(
            f"{API_BASE_URL}/vouchers/admin/{voucher_id}",
            headers=headers
        )
        print("Get voucher response:", get_response.status_code)
        
        # Test update voucher
        update_data = {"description": "Updated test voucher"}
        update_response = requests.put(
            f"{API_BASE_URL}/vouchers/admin/{voucher_id}",
            headers=headers,
            json=update_data
        )
        print("Update voucher response:", update_response.status_code)
        
        # Test delete voucher
        delete_response = requests.delete(
            f"{API_BASE_URL}/vouchers/admin/{voucher_id}",
            headers=headers
        )
        print("Delete voucher response:", delete_response.status_code)
    
    # Test listing vouchers with filters
    list_response = requests.get(
        f"{API_BASE_URL}/vouchers?page=1&limit=10&voucher_type=discount",
        headers=headers
    )
    print("List vouchers response:", list_response.status_code)
    
    return {
        "create": create_response.status_code == 201,
        "get": get_response.status_code == 200 if 'get_response' in locals() else False,
        "update": update_response.status_code == 200 if 'update_response' in locals() else False,
        "delete": delete_response.status_code == 200 if 'delete_response' in locals() else False,
        "list": list_response.status_code == 200
    }

# Run voucher management tests
test_results = test_voucher_management()
print("\nTest Results:", json.dumps(test_results, indent=2))

# Order Management Testing

Now let's test the order management endpoints:
- GET /orders (list all orders)
- POST /orders/:id/zalopay-refund
- POST /orders/:id/payos-refund

We'll create test cases for order management and refund operations:

In [ ]:
def test_order_management():
    # Test listing orders
    list_response = requests.get(
        f"{API_BASE_URL}/orders?page=1&limit=10",
        headers=headers
    )
    print("List orders response:", list_response.status_code)
    
    if list_response.status_code == 200:
        orders = list_response.json()
        if orders and len(orders) > 0:
            test_order_id = orders[0].get('_id')
            
            # Test ZaloPay refund
            zalopay_refund_response = requests.post(
                f"{API_BASE_URL}/orders/{test_order_id}/zalopay-refund",
                headers=headers
            )
            print("ZaloPay refund response:", zalopay_refund_response.status_code)
            
            # Test PayOS refund
            payos_refund_response = requests.post(
                f"{API_BASE_URL}/orders/{test_order_id}/payos-refund",
                headers=headers
            )
            print("PayOS refund response:", payos_refund_response.status_code)
    
    return {
        "list": list_response.status_code == 200,
        "zalopay_refund": zalopay_refund_response.status_code == 200 if 'zalopay_refund_response' in locals() else False,
        "payos_refund": payos_refund_response.status_code == 200 if 'payos_refund_response' in locals() else False
    }

# Run order management tests
order_test_results = test_order_management()
print("\nOrder Test Results:", json.dumps(order_test_results, indent=2))

# UI/UX Testing Checklist

Below is a checklist for manual UI/UX testing of the admin interface. These tests should be performed in the browser:

## Voucher Management UI
- [ ] Voucher list loads and displays correctly
- [ ] Pagination works smoothly
- [ ] Filtering by voucher type works
- [ ] Search functionality works
- [ ] Create voucher form validates input correctly
- [ ] Date picker works for start/end dates
- [ ] Edit voucher loads existing data correctly
- [ ] Delete confirmation dialog works
- [ ] Responsive design works on mobile/tablet

## Order Management UI
- [ ] Order list loads with proper pagination
- [ ] Order details display correctly
- [ ] Refund buttons are properly placed
- [ ] Refund confirmation works
- [ ] Order status updates in real-time
- [ ] Filter by date range works
- [ ] Search by order ID works
- [ ] Responsive design works on mobile/tablet

## General UI/UX
- [ ] Navigation is intuitive
- [ ] Loading states are shown appropriately
- [ ] Error messages are clear and helpful
- [ ] Success messages are displayed
- [ ] Session handling works (logout/login)
- [ ] Breadcrumbs show correct location
- [ ] All forms have proper validation
- [ ] All tables have proper sorting

## Improvement Suggestions
1. Add bulk actions for vouchers (bulk delete/update)
2. Add export functionality for orders and vouchers
3. Implement real-time notifications for new orders
4. Add quick filters for common scenarios
5. Implement dark mode support
6. Add keyboard shortcuts for common actions